# Question 20 - 14. Longest Common Prefix

Write a function to find the longest common prefix string amongst an array of strings.

If there is no common prefix, return an empty string `""`.

**Example 1:**

    Input: strs = ["flower","flow","flight"]
    Output: "fl"

**Example 2:**

    Input: strs = ["dog","racecar","car"]
    Output: ""
    Explanation: There is no common prefix among the input strings.

**Constraints:**

- `1 <= strs.length <= 200`
- `0 <= strs[i].length <= 200`
- `strs[i]` consists of only lowercase English letters if it is non-empty.

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Vertical scan: compare column by column, stop at the first mismatch

📘 **Revise first:** [Part 4 · strings](./0-concepts-array-and-string.ipynb) · [Part 5.3 · startswith](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (try every prefix length) | O(n · m²) | O(m) |
| 2️⃣ Optimized (vertical scan) | O(S) | O(1) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

Find the longest starting piece that **every** word shares.

```
["flower", "flow", "flight"]  → "fl"
["dog", "racecar", "car"]     → ""   (nothing in common at the start)
```

**Everyday picture: stacking the words like rows in a spreadsheet.** Write each word on its own row, left-aligned, and read **down each column**:

```
column:  0 1 2 3 4 5
         f l o w e r
         f l o w
         f l i g h t
         ✅ ✅ ❌
```
Column 0: all `f` ✅. Column 1: all `l` ✅. Column 2: `o`, `o`, `i` ❌, stop. The prefix is everything before the first bad column: **"fl"**.

That's the whole algorithm. We stop at the **first** disagreement, or as soon as **any word runs out of letters**.

### Step 0 · Clarify before coding

🗣️ *"I want the longest string that every word starts with. It can't be longer than the shortest word, and it must be a prefix of the first word, so I'll use the first word as my reference."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Can a word be empty? | **Yes.** | Then the answer is `""`. |
| Only one word? | The answer is that word. | |
| Lowercase only? | **Yes.** | No case folding. |

**Notation:** `n` = number of words, `m` = length of the shortest word, `S` = total number of characters across all words.

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** take the first word. Try its **longest** prefix (the whole word), then one shorter, then shorter again… For each candidate, check whether **every** word starts with it. Return the first one that passes.

🗣️ *"The naive approach tries candidate prefixes from longest to shortest and checks each against every word with startswith. That can be about m candidates × n words × m characters per check, roughly O(n·m²). The waste is that I re-check the same leading characters for every candidate length."*

In [1]:
class SolutionBrute:
    def longestCommonPrefix(self, strs: list[str]) -> str:
        first = strs[0]
        for length in range(len(first), -1, -1):        # longest candidate first
            prefix = first[:length]
            if all(s.startswith(prefix) for s in strs):
                return prefix
        return ""

### Step 2 · Optimize: the "aha" moment

> 💡 **Aha:** check the words **one column at a time** (the spreadsheet picture). For column `i`, take the first word's letter and confirm every other word has the **same letter at the same position**. The moment one word is **too short** or **differs**, the answer is the first word up to (not including) column `i`. Each character gets compared **at most once**.

🗣️ *"I'll scan column by column. For each position i in the first word, I check that every other word is long enough and has the same character at i. At the first failure, I return the first word's prefix up to i. If I get through the whole first word, it's the answer. Each character is compared at most once, so it's O(S) time and O(1) extra space."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Try every prefix length | ❌ | Re-checks the same letters over and over. |
| Horizontal: start with word 1, trim it against each word in turn | ✅ | Also O(S). Can do extra work if the first word is long and a later one is short. |
| **Vertical: column by column** | ✅ **best** | Stops at the **first** bad column across all words. Never reads past the answer + 1. |
| Sort the words, compare only the first and last | ⚠️ | Clever (the most different words end up at the two ends), but sorting costs O(S log n). Using `min()` and `max()` instead of sorting brings it back to O(S). |
| Build a **trie** (letter tree) of all words | ❌ | O(S) time but O(S) memory. Only worth it for many prefix questions on the same words. |
| Binary search on the prefix length | ⚠️ | O(S log m). No better than vertical. |

In [2]:
class Solution:
    def longestCommonPrefix(self, strs: list[str]) -> str:
        first = strs[0]
        for i, ch in enumerate(first):          # each column of the first word
            for s in strs[1:]:
                if i == len(s) or s[i] != ch:   # word too short, or letters differ
                    return first[:i]
        return first                            # the whole first word is shared

### Step 3 · Toy example walkthrough (tell it like a story)

`strs = ["interview", "internet", "interval"]`

| column i | "interview"[i] | "internet"[i] | "interval"[i] | all equal? |
|---|---|---|---|---|
| 0 | i | i | i | ✅ |
| 1 | n | n | n | ✅ |
| 2 | t | t | t | ✅ |
| 3 | e | e | e | ✅ |
| 4 | r | r | r | ✅ |
| 5 | v | **n** | v | ❌ → return `"inter"` |

**Narrated:** "Column by column: i, n, t, e, r all match. At column 5 the first word has 'v' but 'internet' has 'n', so the shared prefix stops there: 'inter'."

**`["dog", "racecar", "car"]`** → column 0: `d` vs `r` ❌ → `""` immediately.

In [3]:
cases = [
    (["flower", "flow", "flight"], "fl"),
    (["dog", "racecar", "car"], ""),
    (["alone"], "alone"),
    (["", "abc"], ""),
    (["abc", ""], ""),
    (["ab", "a"], "a"),                 # first word is the longer one
    (["same", "same", "same"], "same"),
    (["interview", "internet", "interval"], "inter"),
]
for strs, expected in cases:
    assert SolutionBrute().longestCommonPrefix(strs) == expected
    assert Solution().longestCommonPrefix(strs) == expected
    print(strs, "->", repr(expected))
print("All tests passed ✅")

['flower', 'flow', 'flight'] -> 'fl'
['dog', 'racecar', 'car'] -> ''
['alone'] -> 'alone'
['', 'abc'] -> ''
['abc', ''] -> ''
['ab', 'a'] -> 'a'
['same', 'same', 'same'] -> 'same'
['interview', 'internet', 'interval'] -> 'inter'
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Try every prefix | O(n · m²) | O(m) | Re-checks shared letters for each candidate. |
| Sort + compare the ends | O(S log n) | O(m) | The sort dominates. |
| Trie | O(S) | O(S) | Builds a whole tree of letters. |
| **Vertical scan** | **O(S)** | **O(1)** | Each letter is compared at most once; stops early. |

**Why O(S) time?** We check `n` words per column, for at most `len(answer) + 1` columns. That's never more than the total number of characters, and often far less.

**Why O(1) memory?** Apart from the returned slice (the output), we store only loop indexes.

**Edge cases to mention:** a single word, an empty word anywhere, the first word longer than the others (`["ab", "a"]`), all words identical.

**Likely follow-up:** *"Many queries asking for the shared prefix with a new word?"* → Then a **trie** pays for its build cost.

---

#### 🗣️ Full interview script (about 40 seconds)

*"I'll use the first word as a reference and scan column by column. For each position, I check that every other word is long enough and has the same character there. At the first column where that fails, I return the first word up to that point; if nothing fails, the whole first word is the prefix.*

*Each character is compared at most once and I stop at the first mismatch, so it's O(S) time, where S is the total number of characters, and O(1) extra space."*